# Optimizer Cookbook (PyTorch)

> Goal: a practical, runnable reference for common deep-learning optimizers — what they do, when to use them, and sane starting hyperparameters.

This notebook is organized as:
- Environment check
- Minimal training scaffold (data/model/train loop)
- Optimizer theory + PyTorch configs
- A small comparison harness (optional)

In [3]:
import torch
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))

CUDA available: True
GPU: NVIDIA GeForce RTX 3050 6GB Laptop GPU


---

## Suggested Project Structure (optional)
If you later want to turn this into a full mini-project, this is a clean layout:

```text
optimizer_cookbook/
│
├── data/                          # Auto-downloaded datasets (CIFAR-10, MNIST)
├── models/                        # simple_cnn.py, resnet.py, ...
├── utils/                         # trainer.py, plotter.py, logger.py
├── results/                       # logs/ + plots/
├── notebooks/                     # one notebook per optimizer + comparisons
└── README.md
```

You can also keep everything in this single notebook if you prefer.

---

## Notebook Recipe Template
For each optimizer, keep the same structure so results are comparable:

1. **Theory** — what it is + update rule + intuition
2. **Config** — learning rate, weight decay, betas/momentum, eps, epochs
3. **Model + Data** — same model & dataset across optimizers
4. **Train** — train loop + logs per epoch
5. **Results** — curves + best accuracy/lowest loss
6. **Analysis** — when to use / when not to + observations

---

## Setup (one-time)
This notebook assumes `torch` + `torchvision` are available (see `requirements.txt` in your other experiment folders).

Tip: keep the **model/dataset fixed** while you change optimizers — otherwise comparisons are misleading.

In [5]:
import os
import random
from dataclasses import dataclass
from typing import Dict, Tuple

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

def seed_everything(seed: int = 42) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

seed_everything(42)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
device

device(type='cuda')

In [5]:
@dataclass
class TrainConfig:
    dataset: str = 'cifar10'          # 'cifar10' or 'mnist'
    batch_size: int = 128
    epochs: int = 5                    # keep small for quick comparisons
    lr: float = 1e-3
    weight_decay: float = 0.0
    num_workers: int = 2
    pin_memory: bool = True

cfg = TrainConfig()
cfg

TrainConfig(dataset='cifar10', batch_size=128, epochs=5, lr=0.001, weight_decay=0.0, num_workers=2, pin_memory=True)

In [6]:
from torchvision import datasets, transforms
from torch.utils.data import DataLoader

def make_loaders(dataset: str, batch_size: int, num_workers: int = 2, pin_memory: bool = True) -> Tuple[DataLoader, DataLoader, int]:
    if dataset.lower() == 'mnist':
        transform = transforms.Compose([
            transforms.ToTensor(),
            transforms.Normalize((0.1307,), (0.3081,)),
        ])
        train_ds = datasets.MNIST(root='data', train=True, download=True, transform=transform)
        test_ds = datasets.MNIST(root='data', train=False, download=True, transform=transform)
        num_classes = 10
    elif dataset.lower() == 'cifar10':
        transform_train = transforms.Compose([
            transforms.RandomCrop(32, padding=4),
            transforms.RandomHorizontalFlip(),
            transforms.ToTensor(),
            transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2470, 0.2435, 0.2616)),
        ])
        transform_test = transforms.Compose([
            transforms.ToTensor(),
            transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2470, 0.2435, 0.2616)),
        ])
        train_ds = datasets.CIFAR10(root='data', train=True, download=True, transform=transform_train)
        test_ds = datasets.CIFAR10(root='data', train=False, download=True, transform=transform_test)
        num_classes = 10
    else:
        raise ValueError("dataset must be 'cifar10' or 'mnist'")

    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True, num_workers=num_workers, pin_memory=pin_memory)
    test_loader = DataLoader(test_ds, batch_size=batch_size, shuffle=False, num_workers=num_workers, pin_memory=pin_memory)
    return train_loader, test_loader, num_classes

train_loader, test_loader, num_classes = make_loaders(cfg.dataset, cfg.batch_size, cfg.num_workers, cfg.pin_memory)
len(train_loader), len(test_loader), num_classes

100%|██████████| 170M/170M [15:11<00:00, 187kB/s]    


Extracting data\cifar-10-python.tar.gz to data
Files already downloaded and verified


(391, 79, 10)

In [7]:
class SimpleCNN(nn.Module):
    def __init__(self, in_channels: int, num_classes: int):
        super().__init__()
        self.conv1 = nn.Conv2d(in_channels, 32, 3, padding=1)
        self.conv2 = nn.Conv2d(32, 64, 3, padding=1)
        self.pool = nn.MaxPool2d(2)
        self.dropout = nn.Dropout(0.25)
        self.fc1 = nn.Linear(64 * 8 * 8 if in_channels == 3 else 64 * 7 * 7, 256)
        self.fc2 = nn.Linear(256, num_classes)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = self.pool(F.relu(self.conv1(x)))
        x = self.pool(F.relu(self.conv2(x)))
        x = self.dropout(x)
        x = torch.flatten(x, 1)
        x = F.relu(self.fc1(x))
        x = self.dropout(x)
        return self.fc2(x)

in_channels = 3 if cfg.dataset.lower() == 'cifar10' else 1
model = SimpleCNN(in_channels=in_channels, num_classes=num_classes).to(device)
sum(p.numel() for p in model.parameters())

1070794

In [7]:
# Model visualization (layer shapes + params)
from torchinfo import summary

# Make this cell robust: infer input shape from the model itself, so it works even
# if cfg/in_channels aren't defined in the current kernel session.
in_channels = int(getattr(getattr(model, "conv1", None), "in_channels", 3))

# SimpleCNN uses fc1 input features to distinguish CIFAR10 (64*8*8) vs MNIST (64*7*7)
fc1_in = int(getattr(getattr(model, "fc1", None), "in_features", 64 * 8 * 8))
if fc1_in == 64 * 8 * 8:
    height, width = 32, 32
elif fc1_in == 64 * 7 * 7:
    height, width = 28, 28
else:
    # Fallback: assume 32x32 if shape is unknown
    height, width = 32, 32

input_size = (1, in_channels, height, width)

summary(
    model,
    input_size=input_size,
    col_names=("input_size", "output_size", "num_params", "kernel_size", "mult_adds"),
    row_settings=("var_names",),
    depth=3,
    verbose=0,
 )

NameError: name 'model' is not defined

In [8]:
from time import time

def accuracy_from_logits(logits: torch.Tensor, targets: torch.Tensor) -> float:
    preds = logits.argmax(dim=1)
    return (preds == targets).float().mean().item()

@torch.no_grad()
def evaluate(model: nn.Module, loader: DataLoader) -> Dict[str, float]:
    model.eval()
    total_loss = 0.0
    total_acc = 0.0
    total_n = 0
    criterion = nn.CrossEntropyLoss()
    for x, y in loader:
        x, y = x.to(device), y.to(device)
        logits = model(x)
        loss = criterion(logits, y)
        batch_size = x.size(0)
        total_loss += loss.item() * batch_size
        total_acc += accuracy_from_logits(logits, y) * batch_size
        total_n += batch_size
    return {'loss': total_loss / total_n, 'acc': total_acc / total_n}

def train_one_epoch(model: nn.Module, loader: DataLoader, optimizer: torch.optim.Optimizer) -> Dict[str, float]:
    model.train()
    criterion = nn.CrossEntropyLoss()
    total_loss = 0.0
    total_acc = 0.0
    total_n = 0
    for x, y in loader:
        x, y = x.to(device), y.to(device)
        optimizer.zero_grad(set_to_none=True)
        logits = model(x)
        loss = criterion(logits, y)
        loss.backward()
        optimizer.step()

        batch_size = x.size(0)
        total_loss += loss.item() * batch_size
        total_acc += accuracy_from_logits(logits, y) * batch_size
        total_n += batch_size
    return {'loss': total_loss / total_n, 'acc': total_acc / total_n}

def fit(model: nn.Module, train_loader: DataLoader, test_loader: DataLoader, optimizer: torch.optim.Optimizer, epochs: int) -> Dict[str, list]:
    history = {'train_loss': [], 'train_acc': [], 'test_loss': [], 'test_acc': []}
    for epoch in range(1, epochs + 1):
        t0 = time()
        train_metrics = train_one_epoch(model, train_loader, optimizer)
        test_metrics = evaluate(model, test_loader)
        history['train_loss'].append(train_metrics['loss'])
        history['train_acc'].append(train_metrics['acc'])
        history['test_loss'].append(test_metrics['loss'])
        history['test_acc'].append(test_metrics['acc'])
        dt = time() - t0
        print(f"Epoch {epoch:02d}/{epochs} | train loss {train_metrics['loss']:.4f} acc {train_metrics['acc']:.4f} | test loss {test_metrics['loss']:.4f} acc {test_metrics['acc']:.4f} | {dt:.1f}s")
    return history

---

## Optimizer: the knobs that matter
Most optimizers expose a few recurring hyperparameters:

- **Learning rate ($\eta$)**: the most important knob. If training diverges, lower this first.
- **Momentum / betas**: smooths noisy gradients; can speed convergence.
- **Weight decay**: regularization. Prefer **AdamW** when using weight decay with Adam-like methods.
- **Epsilon ($\epsilon$)**: numerical stability term in adaptive methods (rarely changed).

### Weight decay vs L2 penalty (important)
For SGD, “weight decay” is typically equivalent to adding an L2 penalty. For Adam, classic `Adam(weight_decay=...)` behaves like an L2 penalty *inside* the adaptive update; `AdamW` decouples it (usually preferred).

In [9]:
import torch.optim as optim

def make_optimizer(name: str, params, lr: float, weight_decay: float = 0.0, **kwargs) -> torch.optim.Optimizer:
    """Create a PyTorch optimizer by name with sane defaults. kwargs override defaults."""
    name = name.lower()

    if name == 'sgd':
        # Vanilla SGD: v_{t+1}=v_t - lr * g_t
        return optim.SGD(params, lr=lr, momentum=0.0, weight_decay=weight_decay, nesterov=False, **kwargs)
    if name in {'momentum', 'sgd_momentum'}:
        return optim.SGD(params, lr=lr, momentum=0.9, weight_decay=weight_decay, nesterov=False, **kwargs)
    if name in {'nesterov', 'sgd_nesterov'}:
        return optim.SGD(params, lr=lr, momentum=0.9, weight_decay=weight_decay, nesterov=True, **kwargs)

    if name == 'adagrad':
        # Good for sparse features; LR often needs to be higher than Adam
        return optim.Adagrad(params, lr=lr, weight_decay=weight_decay, **kwargs)
    if name == 'rmsprop':
        return optim.RMSprop(params, lr=lr, alpha=0.99, eps=1e-8, momentum=0.0, centered=False, weight_decay=weight_decay, **kwargs)

    if name == 'adam':
        return optim.Adam(params, lr=lr, betas=(0.9, 0.999), eps=1e-8, weight_decay=weight_decay, **kwargs)
    if name == 'adamw':
        return optim.AdamW(params, lr=lr, betas=(0.9, 0.999), eps=1e-8, weight_decay=weight_decay, **kwargs)
    if name == 'radam':
        return optim.RAdam(params, lr=lr, betas=(0.9, 0.999), eps=1e-8, weight_decay=weight_decay, **kwargs)
    if name == 'nadam':
        return optim.NAdam(params, lr=lr, betas=(0.9, 0.999), eps=1e-8, weight_decay=weight_decay, **kwargs)

    if name == 'adadelta':
        return optim.Adadelta(params, lr=lr, rho=0.9, eps=1e-6, weight_decay=weight_decay, **kwargs)
    if name == 'adamax':
        return optim.Adamax(params, lr=lr, betas=(0.9, 0.999), eps=1e-8, weight_decay=weight_decay, **kwargs)

    if name == 'lbfgs':
        # Requires a closure() in the training loop. Included for completeness.
        return optim.LBFGS(params, lr=lr, **kwargs)

    raise ValueError(f'Unknown optimizer: {name}')

available = ['sgd', 'momentum', 'nesterov', 'adagrad', 'rmsprop', 'adam', 'adamw', 'radam', 'nadam', 'adadelta', 'adamax', 'lbfgs']
available

['sgd',
 'momentum',
 'nesterov',
 'adagrad',
 'rmsprop',
 'adam',
 'adamw',
 'radam',
 'nadam',
 'adadelta',
 'adamax',
 'lbfgs']

# 🧠 Optimizers in Deep Learning

## What is an Optimizer?

In deep learning, an **optimizer** is the algorithm responsible for updating a model’s parameters (weights and biases) in order to minimize the loss function.

### Training Loop

1. The model makes predictions  
2. The loss function computes error  
3. Gradients are calculated using backpropagation  
4. The optimizer updates the parameters  

### Core Update Rule

$$\theta \leftarrow \theta - \eta \cdot \nabla L(\theta)$$

Where:

- $\theta$ = model parameters  
- $\eta$ = learning rate  
- $\nabla L(\theta)$ = gradient of the loss  

---

## Why Optimizers Matter

The choice of optimizer directly affects:

- 🚀 Convergence speed  
- 🎯 Final accuracy  
- ⚖️ Generalisation  
- 🔄 Training stability  

Different optimizers behave differently depending on:

- Dataset (dense vs sparse)  
- Model architecture  
- Learning rate sensitivity  

---

# 🔍 Optimizers Covered

---

## 1. Stochastic Gradient Descent (SGD)

### Idea

The simplest optimizer — updates parameters using the gradient of the current batch.

$$\theta_{t+1} = \theta_t - \eta \, g_t$$

### Characteristics

- Single global learning rate  
- No memory of past gradients  

### Pros

- Simple and efficient  
- Good generalisation  

### Cons

- Slow convergence  
- Sensitive to learning rate  

---

## 2. SGD with Momentum

### Idea

Adds a velocity term to accelerate updates.

$$v_t = \beta v_{t-1} + (1 - \beta) g_t$$

$$\theta_{t+1} = \theta_t - \eta \cdot v_t$$

### Intuition

- Accumulates direction over time  
- Reduces oscillations  

### Pros

- Faster than SGD  
- Smoother updates  

### Cons

- Requires tuning  

---

## 3. Adagrad

### Idea

Adapts learning rate per parameter.

$$G_t = G_{t-1} + g_t^2$$

$$\theta_{t+1} = \theta_t - \frac{\eta}{\sqrt{G_t} + \epsilon} \odot g_t$$

### Intuition

- Frequently updated parameters → smaller LR  
- Rare parameters → larger LR  

### Pros

- Works well for sparse data  

### Cons

- Learning rate keeps shrinking  
- Training may stop early  

---

## 4. RMSprop

### Idea

Uses exponential moving average of squared gradients.

$$E[g^2]_t = \beta E[g^2]_{t-1} + (1 - \beta) g_t^2$$

$$\theta_{t+1} = \theta_t - \frac{\eta}{\sqrt{E[g^2]_t} + \epsilon} \odot g_t$$

### Intuition

- Focuses on recent gradients  
- Prevents LR decay to zero  

### Pros

- Stable and efficient  
- Works well in practice  

### Cons

- No momentum by default  

---

## 5. Adam (Adaptive Moment Estimation)

### Idea

Combines momentum and adaptive learning rates.

$$m_t = \beta_1 m_{t-1} + (1 - \beta_1) g_t$$

$$v_t = \beta_2 v_{t-1} + (1 - \beta_2) g_t^2$$

### Bias Correction

$$\hat{m}_t = \frac{m_t}{1 - \beta_1^t}$$

$$\hat{v}_t = \frac{v_t}{1 - \beta_2^t}$$

### Update Rule

$$\theta_{t+1} = \theta_t - \frac{\eta}{\sqrt{\hat{v}_t} + \epsilon} \odot \hat{m}_t$$

### Pros

- Fast convergence  
- Works well by default  

### Cons

- May generalise worse than SGD  
- Improper weight decay handling  

---

## 6. AdamW

### Idea

Decouples weight decay from gradient updates.

### Update Rule

$$\theta_{t+1} = (1 - \eta \lambda)\,\theta_t - \frac{\eta}{\sqrt{\hat{v}_t} + \epsilon} \odot \hat{m}_t$$

### Intuition

- Applies proper regularisation  
- Keeps weight decay independent  

### Pros

- Better generalisation than Adam  
- More stable  

### Cons

- Slightly more complex  

---

## 7. RAdam (Rectified Adam)

### Idea

Adds variance rectification to stabilise early training.

### Key Insight

- Adam is unstable early due to poor variance estimates  
- RAdam corrects this automatically  

### Behavior

- Acts like SGD in early steps  
- Transitions to Adam later  

### Pros

- No warmup required  
- More stable training  

### Cons

- Slightly more computational overhead  

---

# 🧠 Summary Table

| Optimizer        | Key Feature                  | Best Use Case              |
|------------------|-----------------------------|----------------------------|
| SGD              | Simple gradient descent     | Baseline models            |
| SGD + Momentum   | Faster convergence          | Vision tasks               |
| Adagrad          | Adaptive per parameter      | Sparse data (NLP)          |
| RMSprop          | EMA of gradients            | RNNs, RL                   |
| Adam             | Momentum + adaptive LR      | General purpose            |
| AdamW            | Correct weight decay        | Default choice             |
| RAdam            | Stable early training       | No warmup setups           |

---

## Optimizers (theory + practice)

### 1) SGD
**Update:** $\theta_{t+1} = \theta_t - \eta \, g_t$ where $g_t = \nabla_\theta \mathcal{L}(\theta_t)$.
- Pros: simple, strong generalization (often), works great with good LR schedule
- Cons: sensitive to LR; can be slow without momentum/schedules
- Starting point: `lr=0.1` for CIFAR10 with SGD+momentum (batch 128), or smaller for tiny models

### 2) Momentum SGD
Keeps an exponential moving average of gradients.
$$v_{t+1}=\mu v_t + g_t, \quad \theta_{t+1}=\theta_t-\eta v_{t+1}$$
- Typical: `momentum=0.9`
- Often a strong baseline for CNNs/vision tasks

### 3) Nesterov Momentum (NAG)
Looks ahead before computing the gradient (often slightly better than classical momentum).
- Typical: SGD with `momentum=0.9, nesterov=True`

### 4) AdaGrad
Adapts per-parameter step sizes using accumulated squared gradients.
- Great for sparse features (e.g., NLP with sparse embeddings)
- Can decay learning rate too aggressively in deep nets
- Often needs higher `lr` than Adam

### 5) RMSProp
Uses an EMA of squared gradients to avoid AdaGrad’s aggressive decay.
- Common for RNNs / non-stationary losses
- Typical: `alpha=0.99`, optional momentum

### 6) Adam
Momentum + RMSProp style adaptivity.
- Typical: `lr=1e-3`, `betas=(0.9,0.999)`
- Very robust for many problems, but weight decay handling matters

### 7) AdamW (recommended over Adam when using weight decay)
Decoupled weight decay: applies shrinkage separate from adaptive gradient step.
- Typical: `lr=3e-4` to `1e-3`, `weight_decay=1e-2` for transformers; smaller for small CNNs

### 8) RAdam / NAdam
Variants that can improve stability or warmup behavior.
- `RAdam` can be more stable early training
- `NAdam` combines Nesterov-style momentum with Adam-style adaptivity

### 9) LBFGS (second-order-ish; special case)
Good for some small problems; requires a **closure** function and full-batch-ish behavior.
- Not commonly used for large CNN/transformer training